# 05 — Optional benchmarks: LSTM and TimeGPT

The core of this project deliberately runs on classical and tree-based
models: cheap, robust and sufficient for the demand structure shown in
notebooks 02-04. Both optional benchmarks are nevertheless **fully
implemented** behind guarded imports — this notebook explains what each adds,
what it costs, and runs them when the optional dependencies are present.

| Benchmark | Module | What it could add | What it costs |
| --- | --- | --- | --- |
| LSTM (PyTorch) | `clinic_forecast.models.lstm` | Sequence memory for episode persistence | Heavy dependency, tuning, seed sensitivity |
| TimeGPT (Nixtla) | `clinic_forecast.models.timegpt` | Zero-shot foundation-model reference | External API, per-call cost, data leaves the environment |

Both return the project's common forecast schema
(`clinic_id, date, forecast, model`), so the shared evaluation utilities and
the staffing layer consume them unchanged.

In [1]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
pd.set_option("display.max_columns", 60)

from clinic_forecast.data import generate_network_data

data_path = PROJECT_ROOT / "data" / "processed" / "clinic_daily_usage.csv"
if data_path.exists():
    usage = pd.read_csv(data_path, parse_dates=["date"])
else:
    usage = generate_network_data().usage
    usage["date"] = pd.to_datetime(usage["date"])

## LSTM baseline

`LSTMForecaster` is a small sequence-to-horizon model: per-clinic
standardised sliding windows, a clinic embedding, one LSTM layer, a linear
head emitting the full horizon, chronological validation split and early
stopping. The dataset preparation (`make_sequence_dataset`) is plain NumPy
and unit-tested without torch; windows never cross clinic boundaries.

Framing matters: the LSTM is a benchmark, not the expected winner. Its
honest role is to answer "does sequence memory add anything over engineered
lag features here?" — and on stable weekly-cycle demand the usual answer is
no, which is itself worth demonstrating.

In [2]:
try:
    import torch  # noqa: F401

    from clinic_forecast.evaluation import comparison_table, evaluate_forecasts
    from clinic_forecast.models.baseline import seasonal_naive_forecast
    from clinic_forecast.models.lstm import LSTMForecaster

    cutoff = usage["date"].max() - pd.Timedelta(days=28)
    train, test = usage[usage["date"] <= cutoff], usage[usage["date"] > cutoff]

    lstm = LSTMForecaster(window=28, horizon=28, epochs=15).fit(train)
    lstm_scored = test.merge(lstm.forecast(train), on=["clinic_id", "date"], how="inner")
    naive_scored = test.merge(
        seasonal_naive_forecast(train=train, future=test),
        on=["clinic_id", "date"], how="left",
    )
    print(comparison_table(evaluate_forecasts(
        pd.concat([lstm_scored, naive_scored], ignore_index=True)
    )))
except ImportError:
    print("PyTorch is not installed (optional group: `poetry install --with optional`).")
    print("The LSTM benchmark is skipped; dataset preparation is still unit-tested.")

                   mae    rmse    wape   bias
model                                        
lstm            29.151  40.725  39.379  1.344
seasonal_naive  34.378  49.227  46.440 -1.098


## TimeGPT benchmark

`timegpt_forecast` wraps the Nixtla SDK: optional import, API key from
`NIXTLA_API_KEY` (see `.env.example`), injectable client for tests (the test
suite uses a mock — no real API calls in CI), and output in the common
forecast schema.

Two honest caveats before enabling it:

- **Governance.** The call sends demand series to an external service. Even
  for aggregate, non-patient data, a real healthcare network would route
  that through a data-processing review first.
- **Interpretation.** If zero-shot TimeGPT matched the tuned global model,
  that would question the feature pipeline's value; a large gap supports it.
  Either result is informative, which is what makes it a good benchmark.

In [3]:
from clinic_forecast.models.timegpt import timegpt_available, timegpt_forecast

if timegpt_available():
    cutoff = usage["date"].max() - pd.Timedelta(days=28)
    train, test = usage[usage["date"] <= cutoff], usage[usage["date"] > cutoff]
    tg = timegpt_forecast(train, horizon=28)
    tg_scored = test.merge(tg, on=["clinic_id", "date"], how="inner")
    from clinic_forecast.evaluation import comparison_table, evaluate_forecasts
    print(comparison_table(evaluate_forecasts(tg_scored)))
else:
    print("TimeGPT is not configured. To enable the benchmark:")
    print("  1. poetry install --with optional   (installs the nixtla SDK)")
    print("  2. set NIXTLA_API_KEY               (see .env.example)")

TimeGPT is not configured. To enable the benchmark:
  1. poetry install --with optional   (installs the nixtla SDK)
  2. set NIXTLA_API_KEY               (see .env.example)


## Status and recommendation

Both wrappers ship tested and ready, but the project's accuracy budget is
better spent elsewhere first: notebook 04 showed remaining error is dominated
by episode onsets, which neither an LSTM nor a foundation model can see
coming from the demand series alone. The benchmarks earn a re-run whenever
the feature pipeline changes materially — they are one function call each.